# 06 — Faithfulness and reliance: RQ3 (P4)

Does Qwen rely on the right features, and does the prior win?
The measurements are on the grid's label_diversity demonstration sets and the ID queries (`scripts/run_rq3.py`, `results/v3/synthetic/p4/rq3.parquet`).

- **π_true:** the generator's ground truth (positive for the 3 load-bearing columns, zero otherwise).
- **π_behav:** how much Qwen's correct-label margin drops when one column of the query is replaced by a donor row's value (hot-deck). The accuracy drop is shown alongside.
- **π_self:** Qwen's own ranking of the columns, asked after the same demonstrations.
- **Directional sensitivity:** each load-bearing column is nudged by ±0.5 SD. The response follows the data if the margin moves in the rule's direction, and follows the name if it moves in the name's direction. The demo-following index (DFI) is the share that follows the data.

Mechanistic probes (P6) will be added to this notebook later.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.utils.config import load_config, resolve_path  # first src import: sets the thread and MPS settings

config = load_config()

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.suites import load_suite
from src.evaluation.analysis import (behavioural_importance, correctness_rho, dfi, directional_sensitivity,
                                     self_report_rho, true_importance_by_column)

RESULTS = resolve_path(config.paths.results)
rq3 = pd.read_parquet(RESULTS / "p4" / "rq3.parquet")
ranks = pd.read_parquet(RESULTS / "p4" / "rq3_rankings.parquet")
manifest = load_suite("eval", config)
truth = true_importance_by_column(manifest)
print(f"{rq3.unit_key.nunique()} units; namings {sorted(rq3.naming.unique())}; seeds {sorted(rq3.seed.unique())}")

## 1. Which columns does Qwen rely on? (π_behav by column role)

In [ ]:
bi = behavioural_importance(rq3)
t = bi.groupby(["naming", "feature_role"])[["margin_drop", "accuracy_drop"]].mean().unstack("naming")
display(t.round(3))
fig, ax = plt.subplots(figsize=(8, 4))
bi.groupby(["feature_role", "naming"])["margin_drop"].mean().unstack("naming").plot.bar(ax=ax, rot=0)
ax.axhline(0, color="black", lw=0.8); ax.set_ylabel("drop in correct-label margin"); ax.set_title("pi_behav by column role")
plt.tight_layout()

## 2. Correctness: ρ(π_true, π_behav) by naming

In [ ]:
rho = correctness_rho(rq3, truth)
display(rho.groupby("naming")["rho"].describe().round(3))
rho_acc = correctness_rho(rq3, truth, value="accuracy_drop")
print("accuracy-drop version:", rho_acc.groupby("naming")["rho"].mean().round(3).to_dict())

## 3. Directional sensitivity and the demo-following index

In [ ]:
d = dfi(rq3)
display(d.groupby("naming")[["follows_data", "follows_name"]].mean().round(3))
ds = directional_sensitivity(rq3)
lb = ds[ds.feature_role == "load_bearing"]
display(lb.groupby(["naming", "data_direction"])["follows_data"].mean().unstack("data_direction").round(3))
f8 = ds[ds.feature_role == "spurious"]
print("f8 nudges: share of responses moving in the task's spurious direction, by naming:",
      f8.groupby("naming")["follows_data"].mean().round(3).to_dict())

Under flipped names, following the data and following the name are mutually exclusive: the name's direction is the opposite of the rule's.
The table by data direction shows the asymmetry found in P3: the conflict is stronger where a positive-sounding name sits on a feature whose larger values lower the label.

## 4. Self-report: π_self against π_behav

In [ ]:
print("names listed per ranking:", ranks.groupby("naming")["n_listed"].describe().round(2).to_dict("index"))
sr = self_report_rho(ranks, rq3)
display(sr.groupby("naming")["rho"].describe().round(3))
with pd.option_context("display.max_colwidth", 200):
    display(ranks[["naming", "task_id", "response"]].head(6))